# Automatic lung-nodule segmentation (L4)

This notebook downloads one public, de-identified LIDC chest CT that has radiologist annotations. It runs the `lung_nodules` specialist, rejects an empty prediction, and creates one ZIP containing the CT, AI DICOM SEG, and an expert DICOM SEG for OHIF.

Before starting: **Runtime → Change runtime type → L4 GPU**. No DICOM upload or `DICOM_INPUT` is needed for this benchmark. This is experimental evaluation, not diagnosis.

In [ ]:
from pathlib import Path
import subprocess, sys

REPO_URL = 'https://github.com/Hamza09Hamza/doctor_assistant.git'
BRANCH = 'classifier-evaluation-colab'
REPO_DIR = Path('/content/doctor_assistant')

if (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
print('Repository commit:', commit)

In [ ]:
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'TotalSegmentator==2.17.0', 'highdicom==0.27.0', 'idc-index==0.12.5'
], check=True)
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > L4 GPU, then reconnect.'
gpu = torch.cuda.get_device_properties(0)
print(f'GPU: {torch.cuda.get_device_name(0)}; VRAM: {gpu.total_memory / 1024**3:.1f} GiB')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/doctor_assistant/lung_nodule_demo')
CACHE_DIR = DRIVE_ROOT / 'idc_cache'
OUTPUT_DIR = DRIVE_ROOT / 'results'
WEIGHTS_DIR = DRIVE_ROOT / 'weights'
WORK_DIR = Path('/content/lung_nodule_work')
for path in (CACHE_DIR, OUTPUT_DIR, WEIGHTS_DIR, WORK_DIR):
    path.mkdir(parents=True, exist_ok=True)
print('Results will persist in:', OUTPUT_DIR)

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-u",
        "-c",
        """
print("Python subprocess started", flush=True)
import torch
print("PyTorch imported", flush=True)
print("CUDA available:", torch.cuda.is_available(), flush=True)
print("GPU:", torch.cuda.get_device_name(0), flush=True)
""",
    ],
    check=True,
)
print("Commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_DIR,
    text=True,
).strip())

print("Script exists:", (
    REPO_DIR / "scripts/run_lidc_lung_nodule_colab.py"
).exists())

In [ ]:
import os

command = [
    sys.executable, '-u', 'scripts/run_lidc_lung_nodule_colab.py',
    '--cache-dir', str(CACHE_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--work-dir', str(WORK_DIR),
]
env = {**os.environ, 'PYTHONUNBUFFERED': '1', 'TOTALSEG_WEIGHTS_PATH': str(WEIGHTS_DIR)}
print('Running the automatic lung-nodule specialist. The CT is ~125 MB; model weights are additional.')
print('Command:', ' '.join(command), flush=True)
process = subprocess.Popen(
    command, cwd=REPO_DIR, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end='', flush=True)
return_code = process.wait()
if return_code:
    raise subprocess.CalledProcessError(return_code, command)

In [ ]:
import json
latest = json.loads((OUTPUT_DIR / 'latest_dicom_seg_run.json').read_text())
print('AI segments:', latest['validation']['segment_labels'])
print('AI frames:', latest['validation']['frame_count'])
print('OHIF comparison ZIP:', latest['comparison_bundle'])
print('Keep this ZIP; the local publish command will load CT + AI + expert overlays into OHIF.')

## Put the result into local OHIF

After Colab succeeds, download `ohif_ai_vs_expert_bundle.zip` from the printed Google Drive path. On the laptop, with Orthanc running, execute:

```bash
python scripts/publish_dicom_seg_to_orthanc.py /path/to/ohif_ai_vs_expert_bundle.zip
```

The command prints the exact `http://localhost:3000/...` OHIF URL. Open it and use **SEG LOAD** if the overlays are not hydrated automatically.